# Kubo-Ando Geometric Mean of Positive Definite Matrices

## Overview

The set of **symmetric positive definite (SPD) matrices** carries a rich Riemannian geometry that goes far beyond the flat Euclidean structure. In this geometry, the natural notion of "midpoint" between two SPD matrices $A$ and $B$ is not their arithmetic mean $\frac{A+B}{2}$, but their **geometric (Riemannian) mean** $A \# B$.

This distinction is not merely academic. The geometric mean:
- preserves the positivity of matrices under inversion and congruence transformations,
- arises naturally in quantum information (Kubo-Ando operator means),
- plays a key role in diffusion tensor imaging, covariance interpolation, and matrix signal processing.

## The SPD Manifold

The space $\mathcal{P}_n$ of $n \times n$ SPD matrices is an open convex cone in $\text{Sym}_n(\mathbb{R})$. It carries a natural Riemannian metric at $A \in \mathcal{P}_n$:
$$
\langle U, V \rangle_A = \mathrm{tr}(A^{-1} U A^{-1} V)
$$
which induces the geodesic distance:
$$
d(A, B) = \|\log(A^{-1/2} B A^{-1/2})\|_F = \left(\sum_i \log^2 \lambda_i\right)^{1/2}
$$
where $\lambda_i$ are the eigenvalues of $A^{-1/2} B A^{-1/2}$.

## Geodesic and Geometric Mean

The unique geodesic connecting $A$ to $B$ in $\mathcal{P}_n$ is parametrized by $t \in [0, 1]$:
$$
A \#_t B = A^{1/2} \left(A^{-1/2} B A^{-1/2}\right)^t A^{1/2}
$$
At $t = 1/2$ this gives the **geometric mean** (geodesic midpoint):
$$
A \# B = A^{1/2} \left(A^{-1/2} B A^{-1/2}\right)^{1/2} A^{1/2}
$$

## Key Properties

- **Congruence invariance**: $P(A \# B)P^\top = (PAP^\top) \# (PBP^\top)$ for invertible $P$
- **Operator monotone**: $A \leq A' \Rightarrow A \# B \leq A' \# B$
- **Operator concave**: $A \# B \geq \lambda (A_1 \# B_1) + (1-\lambda)(A_2 \# B_2)$ for convex combinations
- **AGM inequality** (operator version):
$$
\left(\frac{A^{-1} + B^{-1}}{2}\right)^{-1} \leq A \# B \leq \frac{A+B}{2}
$$

## What This Notebook Demonstrates

We represent $2 \times 2$ SPD matrices as **ellipses** (level sets of $x^\top A^{-1} x = 1$). We visualize:
1. The geometric vs arithmetic mean paths between two SPD matrices
2. The full geodesic $A \#_t B$ swept over $t \in [0,1]$
3. How the geometric mean preserves shape structure better than the arithmetic mean

### Setup

We use `numpy` for linear algebra and `scipy.linalg.fractional_matrix_power` for computing matrix powers $A^t$ for non-integer $t$. The ellipse corresponding to an SPD matrix $A$ is the set $\{x : x^\top A^{-1} x = 1\}$, which has semi-axes given by $\sqrt{\lambda_i}$ in the eigenvector directions.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from scipy.linalg import fractional_matrix_power

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

### Representing SPD Matrices as Ellipses

A $2 \times 2$ SPD matrix $A$ has the eigendecomposition $A = Q \Lambda Q^\top$ with $\Lambda = \mathrm{diag}(\lambda_1, \lambda_2)$. The ellipse $\{x : x^\top A^{-1} x = 1\}$ has:
- semi-axes of length $\sqrt{\lambda_1}$ and $\sqrt{\lambda_2}$
- orientation given by the columns of $Q$

The function below extracts these parameters to draw the ellipse using matplotlib's `Ellipse` patch.

In [2]:
def spd_to_ellipse(A, center=(0, 0), color='blue', alpha=0.3, lw=2, label=None):
    """Return a matplotlib Ellipse patch representing the SPD matrix A."""
    eigvals, eigvecs = np.linalg.eigh(A)
    # semi-axes are sqrt of eigenvalues
    width = 2 * np.sqrt(eigvals[0])
    height = 2 * np.sqrt(eigvals[1])
    # angle of the first eigenvector
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    ellipse = Ellipse(xy=center, width=width, height=height, angle=angle,
                      facecolor=color, edgecolor=color, alpha=alpha,
                      linewidth=lw, fill=True, label=label)
    return ellipse


def spd_geodesic_midpoint(A, B, t=0.5):
    """
    Compute the geodesic interpolant A #_t B.
    Formula: A^{1/2} * (A^{-1/2} B A^{-1/2})^t * A^{1/2}
    """
    A_half = fractional_matrix_power(A, 0.5)
    A_inv_half = fractional_matrix_power(A, -0.5)
    M = A_inv_half @ B @ A_inv_half
    M_t = fractional_matrix_power(M, t)
    result = A_half @ M_t @ A_half
    # symmetrize to handle floating point drift
    return (result + result.T) / 2


def arithmetic_mean_path(A, B, t):
    """Arithmetic (Euclidean) interpolation: (1-t)*A + t*B."""
    return (1 - t) * A + t * B


print("Helper functions defined.")

Helper functions defined.


### Defining Two SPD Matrices

We construct two SPD matrices $A$ and $B$ with notably different shapes and orientations. To ensure they are SPD, we use $A = Q_A \Lambda_A Q_A^\top$ with a rotation matrix $Q_A$ and positive diagonal $\Lambda_A$.

Then we verify the geodesic distance:
$$
d(A, B) = \left\|\log\!\left(A^{-1/2} B A^{-1/2}\right)\right\|_F
$$

In [3]:
def rotation_matrix(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


# Matrix A: elongated horizontally
Q_A = rotation_matrix(np.pi / 6)
D_A = np.diag([3.0, 0.5])
A = Q_A @ D_A @ Q_A.T

# Matrix B: elongated at a different angle
Q_B = rotation_matrix(-np.pi / 4)
D_B = np.diag([0.8, 2.5])
B = Q_B @ D_B @ Q_B.T

# Geodesic distance
A_inv_half = fractional_matrix_power(A, -0.5)
M = A_inv_half @ B @ A_inv_half
log_M = np.linalg.matrix_power(np.eye(2), 0)  # placeholder
eigvals_M = np.linalg.eigvalsh(M)
geo_dist = np.sqrt(np.sum(np.log(eigvals_M) ** 2))

print(f"Matrix A:\n{A}")
print(f"\nMatrix B:\n{B}")
print(f"\nGeodesic distance d(A,B) = {geo_dist:.4f}")

# Verify positive definiteness
print(f"\nEigenvalues of A: {np.linalg.eigvalsh(A)}")
print(f"Eigenvalues of B: {np.linalg.eigvalsh(B)}")

Matrix A:
[[2.375      1.08253175]
 [1.08253175 1.125     ]]

Matrix B:
[[1.65 0.85]
 [0.85 1.65]]

Geodesic distance d(A,B) = 0.7569

Eigenvalues of A: [0.5 3. ]
Eigenvalues of B: [0.8 2.5]


### Geometric vs Arithmetic Mean Path

We now compute both interpolation paths between $A$ and $B$:

- **Geometric path** (Riemannian geodesic): $A \#_t B = A^{1/2}(A^{-1/2}BA^{-1/2})^t A^{1/2}$
- **Arithmetic path** (Euclidean): $(1-t)A + tB$

Each matrix along the path is displayed as an ellipse. The geometric path preserves the intrinsic SPD geometry — notice that the ellipses rotate and scale smoothly along the manifold geodesic, while the arithmetic path can produce "fat" intermediate shapes that distort the geometry.

In [4]:
n_steps = 7
t_vals = np.linspace(0, 1, n_steps)

# Spacing for display: place ellipses at evenly spaced x positions
x_positions = np.linspace(-3, 3, n_steps)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

titles = ["Geometric path (Riemannian geodesic)  $A \\sharp_t B$",
          "Arithmetic path (Euclidean)  $(1-t)A + tB$"]
colors_geo = plt.cm.plasma(np.linspace(0.1, 0.9, n_steps))
colors_arith = plt.cm.viridis(np.linspace(0.1, 0.9, n_steps))

for ax, title, path_fn, cmap_colors in zip(
        axes, titles,
        [spd_geodesic_midpoint, arithmetic_mean_path],
        [colors_geo, colors_arith]):

    for i, (t, xc) in enumerate(zip(t_vals, x_positions)):
        M_t = path_fn(A, B, t)
        ell = spd_to_ellipse(M_t, center=(xc, 0),
                             color=cmap_colors[i], alpha=0.6, lw=2)
        ax.add_patch(ell)
        ax.text(xc, -1.8, f"t={t:.2f}", ha='center', fontsize=9, color='black')

    ax.set_xlim(-5, 5)
    ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=13)
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.set_xlabel("Interpolation parameter $t$")

    # Mark A and B endpoints
    ax.text(x_positions[0], 2.0, '$A$', ha='center', fontsize=13, fontweight='bold', color='purple')
    ax.text(x_positions[-1], 2.0, '$B$', ha='center', fontsize=13, fontweight='bold', color='darkorange')

plt.suptitle("Geodesic vs Arithmetic Interpolation on $\\mathcal{P}_2$", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig("geo_arith_path.png", dpi=80, bbox_inches='tight')
plt.show()

/var/folders/c3/8qf_y_jj6393y3l0dl0bb3k80000gp/T/ipykernel_51623/1040688945.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### AGM Inequality Verification

The operator arithmetic-geometric-harmonic mean inequality states:
$$
H(A,B) \leq A \# B \leq \frac{A+B}{2}
$$
where the harmonic mean is $H(A,B) = 2(A^{-1} + B^{-1})^{-1} = 2A(A+B)^{-1}B$.

The ordering $X \leq Y$ for SPD matrices means $Y - X$ is positive semidefinite. We verify this numerically by checking the eigenvalues of the difference matrices.

In [5]:
G = spd_geodesic_midpoint(A, B, 0.5)  # geometric mean
Arith = (A + B) / 2                   # arithmetic mean
Harm = 2 * np.linalg.inv(np.linalg.inv(A) + np.linalg.inv(B))  # harmonic mean

# Check H <= G
diff_HG = G - Harm
diff_GA = Arith - G

eig_HG = np.linalg.eigvalsh(diff_HG)
eig_GA = np.linalg.eigvalsh(diff_GA)

print("AGM Inequality verification")
print("="*40)
print(f"Eigenvalues of (G - H):  {eig_HG}  -> all >= 0? {np.all(eig_HG >= -1e-10)}")
print(f"Eigenvalues of (Arith - G): {eig_GA}  -> all >= 0? {np.all(eig_GA >= -1e-10)}")

print("\nMatrix norms:")
print(f"  ||H||_F = {np.linalg.norm(Harm, 'fro'):.4f}")
print(f"  ||G||_F = {np.linalg.norm(G, 'fro'):.4f}")
print(f"  ||Arith||_F = {np.linalg.norm(Arith, 'fro'):.4f}")
print("\nNote: Frobenius norm ordering is consistent with operator ordering here.")

AGM Inequality verification
Eigenvalues of (G - H):  [0.02235758 0.06856007]  -> all >= 0? True
Eigenvalues of (Arith - G): [0.02304385 0.07138118]  -> all >= 0? True

Matrix norms:
  ||H||_F = 2.6657
  ||G||_F = 2.7318
  ||Arith||_F = 2.8005

Note: Frobenius norm ordering is consistent with operator ordering here.


### Congruence Invariance

One of the most elegant properties of the geometric mean is **congruence invariance**:
$$
P(A \# B)P^\top = (PAP^\top) \# (PBP^\top)
$$
for any invertible matrix $P$. This means the geometric mean commutes with changes of coordinates, making it truly intrinsic to the geometry of SPD matrices.

In contrast, the arithmetic mean does **not** satisfy this: $(PAP^\top + PBP^\top)/2 = P \frac{A+B}{2} P^\top$ holds trivially only when $P$ is orthogonal.

We verify congruence invariance numerically for a random invertible $P$.

In [6]:
rng = np.random.default_rng(42)
P = rng.standard_normal((2, 2)) + 2 * np.eye(2)  # random invertible matrix

# LHS: P * (A # B) * P^T
G_AB = spd_geodesic_midpoint(A, B, 0.5)
lhs = P @ G_AB @ P.T

# RHS: (P A P^T) # (P B P^T)
PA = P @ A @ P.T
PB = P @ B @ P.T
rhs = spd_geodesic_midpoint(PA, PB, 0.5)

print("Congruence invariance check:")
print(f"  LHS = P(A#B)P^T:\n{lhs}")
print(f"  RHS = (PAP^T)#(PBP^T):\n{rhs}")
print(f"  Max difference: {np.max(np.abs(lhs - rhs)):.2e}")
print(f"  Congruence invariance holds: {np.allclose(lhs, rhs, atol=1e-8)}")

Congruence invariance check:
  LHS = P(A#B)P^T:
[[ 7.4086102   5.03011889]
 [ 5.03011889 16.76871869]]
  RHS = (PAP^T)#(PBP^T):
[[ 7.4086102   5.03011889]
 [ 5.03011889 16.76871869]]
  Max difference: 4.26e-14
  Congruence invariance holds: True


### Full Geodesic Visualization

We now produce a comprehensive figure showing the full geodesic path $A \#_t B$ for $t \in [0,1]$ alongside the arithmetic path, with additional panel showing the Frobenius norms along each path.

The **Frobenius norm** $\|M\|_F = \sqrt{\mathrm{tr}(M^\top M)}$ is not a Riemannian distance but gives a rough sense of matrix "size". The geometric path has norms varying smoothly according to the geodesic, while the arithmetic path takes the Euclidean shortcut.

In [7]:
t_dense = np.linspace(0, 1, 50)
geo_norms = []
arith_norms = []
det_geo = []
det_arith = []

for t in t_dense:
    G_t = spd_geodesic_midpoint(A, B, t)
    A_t = arithmetic_mean_path(A, B, t)
    geo_norms.append(np.linalg.norm(G_t, 'fro'))
    arith_norms.append(np.linalg.norm(A_t, 'fro'))
    det_geo.append(np.linalg.det(G_t))
    det_arith.append(np.linalg.det(A_t))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_dense, geo_norms, 'b-', lw=2, label='Geometric path')
axes[0].plot(t_dense, arith_norms, 'r--', lw=2, label='Arithmetic path')
axes[0].set_xlabel('$t$')
axes[0].set_ylabel('Frobenius norm')
axes[0].set_title('$\\|M_t\\|_F$ along the path')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_dense, det_geo, 'b-', lw=2, label='Geometric path')
axes[1].plot(t_dense, det_arith, 'r--', lw=2, label='Arithmetic path')
axes[1].set_xlabel('$t$')
axes[1].set_ylabel('Determinant')
axes[1].set_title('$\\det(M_t)$ along the path')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Comparison of Geometric vs Arithmetic Interpolation Properties", fontsize=13)
plt.tight_layout()
plt.savefig("path_properties.png", dpi=80, bbox_inches='tight')
plt.show()
print("Note: det(A#B) = sqrt(det(A)*det(B)) — the geometric mean of determinants.")

Note: det(A#B) = sqrt(det(A)*det(B)) — the geometric mean of determinants.


/var/folders/c3/8qf_y_jj6393y3l0dl0bb3k80000gp/T/ipykernel_51623/3410216444.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interactive Geodesic Explorer

The widget below lets you drag a slider to explore the geodesic $A \#_t B$ for any $t \in [0,1]$. The left panel shows the ellipse at parameter $t$ (geometric mean path), and the right panel shows the arithmetic interpolant at the same $t$. Observe how the geometric path rotates the ellipse more smoothly while the arithmetic path creates an intermediary with larger/rounder shape.

### Static Snapshot

The static cell below generates the representative figure used as the snippet image. It shows both $A$ and $B$ (faint outlines), the geometric mean $A \# B$ (solid), and the arithmetic mean $(A+B)/2$ (dashed), all as ellipses centered at the origin.

In [8]:
STATIC_SNAPSHOT = True

if STATIC_SNAPSHOT:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    n_geo = 9
    t_snap = np.linspace(0, 1, n_geo)
    x_snap = np.linspace(-3.5, 3.5, n_geo)
    colors_snap = plt.cm.coolwarm(np.linspace(0.05, 0.95, n_geo))

    fig_s, ax_s = plt.subplots(figsize=(10, 4))
    ax_s.set_facecolor('#f8f8f8')
    fig_s.patch.set_facecolor('#f8f8f8')

    for i, (t, xc) in enumerate(zip(t_snap, x_snap)):
        G_t = spd_geodesic_midpoint(A, B, t)
        ell = spd_to_ellipse(G_t, center=(xc, 0),
                             color=colors_snap[i], alpha=0.7, lw=1.5)
        ax_s.add_patch(ell)
        if i in [0, n_geo // 2, n_geo - 1]:
            lbl = ['$A$', '$A\\sharp B$', '$B$'][{0: 0, n_geo//2: 1, n_geo-1: 2}[i]]
            ax_s.text(xc, -1.7, lbl, ha='center', fontsize=12, fontweight='bold')

    ax_s.set_xlim(-5, 5)
    ax_s.set_ylim(-2.2, 2.2)
    ax_s.set_aspect('equal')
    ax_s.axis('off')
    ax_s.set_title("Riemannian Geodesic $A \\sharp_t B$ on $\\mathcal{P}_2$",
                   fontsize=14, pad=10)

    plt.tight_layout()
    plt.savefig("snippet.png", dpi=100, bbox_inches='tight',
                facecolor=fig_s.get_facecolor())
    plt.close()
    print("snippet.png saved.")

snippet.png saved.


## Takeaways

- The space of SPD matrices $\mathcal{P}_n$ is a Riemannian manifold with a rich non-Euclidean geometry; the natural geodesic distance is $d(A,B) = \|\log(A^{-1/2}BA^{-1/2})\|_F$.
- The **geometric mean** $A \# B = A^{1/2}(A^{-1/2}BA^{-1/2})^{1/2}A^{1/2}$ is the Riemannian midpoint and satisfies elegant properties such as congruence invariance and the AGM inequality.
- The **arithmetic mean** $(A+B)/2$ and the **harmonic mean** bracket the geometric mean as matrices: $H(A,B) \leq A\#B \leq (A+B)/2$.
- Visualizing SPD matrices as ellipses makes the geometry concrete: the geodesic path smoothly rotates and scales the ellipse shape, unlike the arithmetic path which can produce artificial distortions.
- The determinant along the geometric path satisfies $\det(A\#_tB) = \det(A)^{1-t} \det(B)^t$, a continuous geometric interpolation of determinants.

## Bibliography

- **Kubo, F. & Ando, T.** (1980). *Means of positive linear operators.* Mathematische Annalen, 246(3), 205–224.
- **Bhatia, R.** (2007). *Positive Definite Matrices.* Princeton University Press.
- **Moakher, M.** (2005). *A differential geometric approach to the geometric mean of symmetric positive-definite matrices.* SIAM Journal on Matrix Analysis and Applications, 26(3), 735–747.
- **Arsigny, V., Fillard, P., Pennec, X., Ayache, N.** (2007). *Geometric means in a novel vector space structure on symmetric positive-definite matrices.* SIAM Journal on Matrix Analysis and Applications, 29(1), 328–347.
- **Pennec, X., Fillard, P., Ayache, N.** (2006). *A Riemannian framework for tensor computing.* International Journal of Computer Vision, 66(1), 41–66.